# k-NN — практическое задание

В этом ноутбуке рассматривается метод **k ближайших соседей** (**k-NN**, *k-Nearest Neighbors*) на примере реального набора данных с Kaggle.
Ноутбук рассчитан на выполнение в **Google Colab**.


## Теоретическое введение

### 1. Постановка задачи классификации

Пусть имеется обучающая выборка
$$
\{(x_i, y_i)\}_{i=1}^n, \qquad x_i \in \mathbb{R}^d,
$$
где:
- $x_i$ — вектор признаков объекта;
- $y_i$ — метка класса.

Задача классификации состоит в том, чтобы по признакам нового объекта $x$ определить, к какому классу он принадлежит.

### 2. Идея метода k ближайших соседей

Метод k-NN относится к числу **непараметрических** методов машинного обучения.

В рамках параметрических методов задаётся модель в виде аналитического выражения
$$
z=f(x; w)
$$
и в ходе обучения подбираются наилучшие значения параметров $w$. Непараметрические методы не требуют задать исходную форму модели. Вместо этого при классификации нового объекта алгоритм k-NN:
1. вычисляет расстояния от него до всех объектов обучающей выборки;
2. выбирает $k$ ближайших соседей;
3. относит объект к тому классу, который чаще всего встречается среди этих соседей.

Если обозначить множество $k$ ближайших соседей объекта $x$ через $N_k(x)$, то предсказание можно записать как
$$
\hat y(x)=\mathrm{mode}\{y_i : x_i \in N_k(x)\}.
$$
где $\mathrm{mode}()$ - мода распределения, т.е., значение, которое встречается наиболее часто в наборе данных, «типичный» элемент. В данном случае имеется наиболее часто встречающееся значение $y_i$ среди ближайших соседей.

### 3. Расстояние между объектами

Ключевую роль в k-NN играет понятие расстояния.
Чаще всего используется **евклидово расстояние**:
$$
\rho(x,z)=\sqrt{\sum_{j=1}^d (x_j-z_j)^2}.
$$

Также могут использоваться:
- манхэттенское расстояние;
- расстояние Минковского;
- другие метрики.

Выбор метрики влияет на результат классификации.

### 4. Почему необходимо масштабирование признаков

Если признаки имеют разные масштабы, то признак с большими числовыми значениями начинает доминировать при вычислении расстояния.

Например, если один признак изменяется в диапазоне от 0 до 1, а другой — от 0 до 1000, то вклад второго признака в расстояние будет значительно больше.

Поэтому перед применением k-NN обычно выполняют **стандартизацию**:
$$
x_j^{\text{scaled}} = \frac{x_j-\mu_j}{\sigma_j}.
$$

Это особенно важно для методов, основанных на расстояниях.

### 5. Выбор числа соседей $k$

Параметр $k$ управляет сложностью модели:
- малое значение $k$ делает модель чувствительной к шуму;
- большое значение $k$ делает модель более сглаженной.

При:
- $k=1$ модель может переобучаться;
- слишком большом $k$ модель может терять способность различать классы.

На практике значение $k$ подбирают экспериментально.

### 6. Взвешивание соседей

Иногда ближайшие соседи учитываются не одинаково, а с весами, зависящими от расстояния.
Тогда более близкие объекты сильнее влияют на решение.

В `scikit-learn` это задаётся параметром:
```python
weights="distance"
```

### 7. Оценка качества классификации

Для оценки качества модели классификации часто используют:
- **accuracy** — доля правильных ответов;
- **confusion matrix** — матрица ошибок;
- **precision**, **recall**, **f1-score** — особенно полезны при несбалансированных классах.

### 8. Преимущества и ограничения метода

Преимущества:
- простота идеи;
- отсутствие сложного этапа обучения;
- хорошая работа на простых задачах классификации;
- естественная геометрическая интерпретация.

Ограничения:
- чувствительность к масштабу признаков;
- высокая вычислительная стоимость на больших выборках;
- ухудшение качества в пространствах большой размерности;
- чувствительность к шуму и выбросам.

### 9. Почему этот метод важен

k-NN полезен для изучения, потому что:
- опирается на геометрические идеи;
- показывает роль метрики в машинном обучении;
- даёт понятный базовый алгоритм классификации;
- помогает понять, зачем нужны предобработка данных и масштабирование признаков.


## Набор данных

Для работы используется набор данных о пингвинах Пальмера:
[https://www.kaggle.com/datasets/parulpandey/palmer-archipelago-antarctica-penguin-data](https://www.kaggle.com/datasets/parulpandey/palmer-archipelago-antarctica-penguin-data)

Целевая переменная:
- `species` — вид пингвина.

Примеры признаков:
- `culmen_length_mm` — длина клюва;
- `culmen_depth_mm` — глубина клюва;
- `flipper_length_mm` — длина ласта;
- `body_mass_g` — масса тела;
- `island` — остров;
- `sex` — пол.

Этот датасет хорошо подходит для изучения k-NN, потому что:
- задача является задачей многоклассовой классификации;
- есть несколько наглядных числовых признаков;
- различие между классами удобно анализировать с точки зрения расстояний.


## Как загрузить данные с Kaggle прямо в Google Colab

Выполните предварительную настройку Kaggle и Colab.
1. Откройте Kaggle, зайдите в свой профиль и создайте API-токен.
2. В Colab откройте вкладку **Secrets**.
3. Создайте два секрета:
- `KAGGLE_USERNAME`
- `KAGGLE_KEY`

Не забудьте разрешить доступ к этим секретам из блокнота.

Далее следуйте инструкциям в коде ниже.


In [ ]:
# ШАГ 1. Установка Kaggle API
!pip -q install kaggle

# ШАГ 2. Чтение токена из Colab Secrets
import os
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

# ШАГ 3. Проверка, что всё работает
!kaggle datasets list -s penguin | head

# ШАГ 4. Скачивание датасета
TARGET_DIR = "/content/sci-prog-and-ml"
!mkdir -p {TARGET_DIR}
os.chdir(TARGET_DIR)

# Адрес датасета на Kaggle
DATASET = "parulpandey/palmer-archipelago-antarctica-penguin-data"
!kaggle datasets download -d {DATASET} -p {TARGET_DIR} --unzip

# ШАГ 5. Проверка содержимого папки
print("Содержимое папки после скачивания:")
for file in os.listdir(TARGET_DIR):
    print(" -", file)


## Альтернативный вариант — скачивание файла вручную

- скачайте CSV-файлы с Kaggle вручную;
- поместите их в папку `sci-prog-and-ml` на Google Диске;
- подмонтируйте Google Диск в Colab и перейдите в эту папку.

После этого файлы можно открывать через `pandas.read_csv(...)`.


In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/sci-prog-and-ml')

print("Содержимое текущей папки:")
for file in os.listdir('.'):
    print(" -", file)


## Задание

### Часть 1. Загрузка и первичный анализ данных
1. Найдите CSV-файл с данными о пингвинах и загрузите его в `pandas.DataFrame`.
2. Выведите первые строки таблицы.
3. Определите:
   - число объектов;
   - список признаков;
   - наличие пропущенных значений;
   - типы признаков;
   - число классов в столбце `species`.

### Часть 2. Предобработка данных
1. Выделите целевую переменную `species`.
2. Выберите набор признаков для классификации.
3. Разделите признаки на:
   - числовые;
   - категориальные.
4. Обработайте пропуски.
5. Закодируйте категориальные признаки.
6. Выполните масштабирование числовых признаков.

### Часть 3. Построение модели
1. Разделите выборку на обучающую и тестовую части.
2. Постройте модель `KNeighborsClassifier`.
3. Обучите модель на обучающей выборке.
4. Получите предсказания на тестовой выборке.

### Часть 4. Оценка качества
1. Вычислите accuracy.
2. Постройте confusion matrix.
3. Выведите classification report.
4. Сравните качество модели при нескольких значениях параметра `k`.

### Часть 5. Исследование параметров
1. Проверьте, как меняется качество модели при $k=1, 3, 5, 7, 9, 11$.
2. Сравните варианты:
   - `weights="uniform"`;
   - `weights="distance"`.
3. Сделайте вывод, какое значение `k` оказалось наиболее удачным для данного набора данных.


## Подсказки

### Основные библиотеки
```python
import os
import glob
import numpy as np
import pandas as pd
```

### Как найти нужный CSV-файл

В датасете может быть несколько CSV-файлов. Удобно автоматически выбрать тот, где есть столбец `species`:

```python
csv_files = glob.glob("*.csv")

for path in csv_files:
    tmp = pd.read_csv(path)
    if "species" in tmp.columns:
        df = tmp
        break
```

### Разделение на признаки и целевую переменную
```python
X = df.drop(columns=["species"])
y = df["species"]
```

### Разделение на обучающую и тестовую выборки
```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
```

### Библиотека машинного обучения
Используйте библиотеку `scikit-learn`.

### Модель k-NN
```python
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train, y_train)
```

### Предобработка данных
Для удобной обработки числовых и категориальных признаков используйте:
```python
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
```

### Метрики качества
```python
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
```

### Полезная идея
Удобно построить единый конвейер (`Pipeline`), в котором сначала выполняется предобработка, а затем обучается модель k-NN.


## Решение


In [ ]:
# Поместите Ваш код сюда


## Вопросы для обсуждения

1. Почему для метода k-NN особенно важно масштабирование числовых признаков?
2. Почему этот метод относят к алгоритмам, основанным на расстояниях?
3. Как влияет слишком малое значение `k` на устойчивость модели?
4. Как влияет слишком большое значение `k` на способность модели различать классы?
5. Почему при наличии категориальных признаков требуется их кодирование?


## Дополнительные задания

### Задание 1
Сравните разные метрики расстояния:
- `metric="minkowski"`;
- `metric="manhattan"`;
- `metric="euclidean"`.

### Задание 2
Сравните результаты при:
- `weights="uniform"`;
- `weights="distance"`.

### Задание 3
Постройте график зависимости accuracy от числа соседей `k`.

### Задание 4
Попробуйте обучить модель только на числовых признаках и сравнить результат с вариантом, где используются и категориальные признаки.

### Задание 5
Проанализируйте, какие виды пингвинов чаще всего путает модель, и предложите объяснение.
